# Exploring possible targets for XRISM - 2024

## Import Statements

In [1]:
from astropy.units import Quantity
import astropy.units as u
from astropy.cosmology import FlatLambdaCDM
from astropy.coordinates import SkyCoord
from astropy.table import Table
from astropy.io import fits
import pandas as pd
pd.options.display.max_rows = 999
pd.set_option("max_colwidth", 100)
pd.set_option('display.max_columns', 500)
import numpy as np
import os
from tqdm import tqdm
from matplotlib import pyplot as plt
import warnings

from xga.products import Image
from xga.imagetools.misc import physical_rad_to_pix
from xga.sourcetools.misc import rad_to_ang
from xga.imagetools.profile import annular_mask
from xga.products import ExpMap

%matplotlib inline
warnings.filterwarnings('ignore')

## Define the cosmology

In [2]:
cosmo = FlatLambdaCDM(70, 0.3)

## Read in the sample

In [3]:
samp = pd.read_csv("../../sample_files/lovoccs_southnorth.csv")
samp['LoVoCCS_name'] = samp['LoVoCCSID'].apply(lambda x: 'LoVoCCS-' + str(x))

check = pd.read_csv(("../../../LoVoCCS-Sample/paper_specific_subsamples/proposals/NuSTAR/"
                     "lovoccs_checklist_lite.csv"))

check['Name'] = check['Name'].apply(lambda x: x.replace('RXC ', 'RXC').replace("RBS ", "RBS")
                                    .replace("APMCC ", "APMCC").replace("MKW ", "MKW"))

In [4]:
on_other = pd.merge(samp, check, left_on="other_names", right_on="Name", how="right")
on_alt = pd.merge(samp, check, left_on="alt_name", right_on="Name", how="right")
rel_samp = pd.concat([on_other[~on_other['LoVoCCSID'].isna()], 
                      on_alt[~on_alt['LoVoCCSID'].isna()]]).drop_duplicates('LoVoCCSID')
rel_samp['LoVoCCSID'] = rel_samp['LoVoCCSID'].astype(int)
rel_samp = rel_samp.reset_index(drop=True)

In [5]:
check[~check['Name'].isin(rel_samp.Name)]

,ID,Name,RA,DEC,Redshift,Newly processed,Fully observed,Comments


We reduce the sample to only those clusters which have been fully observed by our optical survey:

In [6]:
rel_samp = rel_samp[rel_samp['Fully observed'] == 'yes'].reset_index(drop=True)
rel_samp['Fully observed'].value_counts()
rel_samp.to_csv('under_consideration.csv', index=False)

In [7]:
rel_samp['sub_samp'].value_counts()

sub_samp
south    79
north     3
Name: count, dtype: int64

In [8]:
rel_samp

,name,MCXC,LoVoCCSID,LoVoCCS_name,ra,dec,redshift,L500,M500,R500,alt_name,other_names,Notes,sub_samp,old_LoVoCCSID,ID,Name,RA,DEC,Redshift,Newly processed,Fully observed,Comments
0,MCXCJ1510.9+0543,J1510.9+0543,1,LoVoCCS-1,227.729167,5.720000,0.0766,8.726709,7.2708,1.3344,A2029,A2029,NaN,south,1.0,1,A2029,227.73,5.72,0.0766,1.0,yes,"with A2033, southern infalling group"
1,MCXCJ0258.9+1334,J0258.9+1334,2,LoVoCCS-2,44.739583,13.579444,0.0739,6.088643,5.8488,1.2421,RXCJ0258.9+1334,A401,L,south,2.0,2,A401,44.74,13.58,0.0739,1.0,yes,Bridge of galaxies between A401/A399 -- similar to gas
2,MCXCJ0041.8-0918,J0041.8-0918,4,LoVoCCS-4,10.458750,-9.301944,0.0555,5.100085,5.3163,1.2103,RXCJ0041.8-0918,A85,"L,losStr",south,4.0,3,A85,10.46,-9.30,0.0555,1.0,yes,NaN
3,MCXCJ2012.5-5649,J2012.5-5649,5,LoVoCCS-5,303.127100,-56.831900,0.0556,4.871933,5.1696,1.1990,RXCJ2012.5-5649,A3667,"L,losStr",south,901.0,4,A3667,303.13,-56.83,0.0556,1.0,yes,NaN
4,MCXCJ2201.9-5956,J2201.9-5956,7,LoVoCCS-7,330.483333,-59.949444,0.0980,4.204419,4.5890,1.1367,RXCJ2201.9-5956,A3827,NaN,south,6.0,5,A3827,330.48,-59.95,0.0980,1.0,yes,"Currently using A3825 catalogs. Note the 2 clusters are at very different redshifts, but RS colo..."
5,MCXCJ0431.4-6126,J0431.4-6126,9,LoVoCCS-9,67.850417,-61.443889,0.0589,3.977333,4.5579,1.1485,RXCJ0431.4-6126,A3266,NaN,south,8.0,6,A3266,67.85,-61.44,0.0589,1.0,yes,NaN
6,MCXCJ1259.3-0411,J1259.3-0411,10,LoVoCCS-10,194.839583,-4.194722,0.0845,3.853573,4.3928,1.1252,RXCJ1259.3-0411,A1651,NaN,south,9.0,7,A1651,194.84,-4.19,0.0845,1.0,yes,NaN
7,MCXCJ0909.1-0939,J0909.1-0939,11,LoVoCCS-11,137.285000,-9.666111,0.0542,3.849660,4.4824,1.1439,RXCJ0909.1-0939,A754,NaN,south,10.0,8,A754,137.28,-9.67,0.0542,1.0,yes,"2 peaks in RS galaxy density map, lensing peak is mainly at BCG"
8,MCXCJ1347.4-3250,J1347.4-3250,12,LoVoCCS-12,206.868333,-32.849722,0.0391,3.819920,4.5067,1.1514,RXCJ1347.4-3250,A3571,L,south,11.0,9,A3571,206.87,-32.85,0.0391,1.0,yes,eRosita
9,MCXCJ0317.9-4414,J0317.9-4414,13,LoVoCCS-13,49.493750,-44.238889,0.0752,3.815928,4.3948,1.1288,RXCJ0317.9-4414,A3112,NaN,south,12.0,10,A3112,49.49,-44.24,0.0752,1.0,yes,"Horologium Supercluster with A3128, A3158, A3266. Xray mass is very close to the lensing mass (S..."


In [9]:
stop

NameError: name 'stop' is not defined

## NuSTAR coverage of the LoVoCCS sample

In [ ]:
field_nustar_cov_frac = []
r500_nustar_cov_frac = []
rvir_nustar_cov_frac = []

for row_ind, row in samp.iterrows():
    cov_path = "../../outputs/coverage_maps/fits/{n}_550pix_20.0arcsec.fits".format(n=row["LoVoCCS_name"])
    cur_cheese = ExpMap(cov_path, '', '', '', '', '', Quantity(0.5, 'keV'), Quantity(2.0, 'keV'))
    with fits.open(cov_path) as covero:
        if row_ind == 0:
            print(covero[0].header['MISSION*'])
            
        cov_arr = covero[0].data
        nustar_cov_arr = cov_arr[4, :, :]
        
        field_nustar_cov_frac.append(nustar_cov_arr.sum() / (nustar_cov_arr.shape[0]*nustar_cov_arr.shape[1]))
        
        pix_cen = Quantity([nustar_cov_arr.shape[0]/2, nustar_cov_arr.shape[1]/2], 'pix').round(0).astype(int)
        rad = Quantity(row['R500'], 'Mpc')
        
        pix_rad = physical_rad_to_pix(cur_cheese, rad, pix_cen, row['redshift'], cosmo)
        
        rad_mask = annular_mask(pix_cen, np.array([0]), np.array([pix_rad.value]), nustar_cov_arr.shape)
        msk_nustar_cov = nustar_cov_arr*rad_mask
        r500_nustar_cov_frac.append(msk_nustar_cov.sum() / rad_mask.sum())
        
        samp.loc[row_ind, 'NUSTAR_R500_FRAC'] = r500_nustar_cov_frac[row_ind]

In [ ]:
field_nustar_cov_frac = []
r500_nustar_cov_frac = []
rvir_nustar_cov_frac = []

for row_ind, row in rel_samp.iterrows():
    cov_path = "../../outputs/coverage_maps/fits/{n}_550pix_20.0arcsec.fits".format(n=row["LoVoCCS_name"])
    cur_cheese = ExpMap(cov_path, '', '', '', '', '', Quantity(0.5, 'keV'), Quantity(2.0, 'keV'))
    with fits.open(cov_path) as covero:
        if row_ind == 0:
            print(covero[0].header['MISSION*'])
            
        cov_arr = covero[0].data
        nustar_cov_arr = cov_arr[4, :, :]
        
        field_nustar_cov_frac.append(nustar_cov_arr.sum() / (nustar_cov_arr.shape[0]*nustar_cov_arr.shape[1]))
        
        pix_cen = Quantity([nustar_cov_arr.shape[0]/2, nustar_cov_arr.shape[1]/2], 'pix').round(0).astype(int)
        rad = Quantity(row['R500'], 'Mpc')
        
        pix_rad = physical_rad_to_pix(cur_cheese, rad, pix_cen, row['redshift'], cosmo)
        
        rad_mask = annular_mask(pix_cen, np.array([0]), np.array([pix_rad.value]), nustar_cov_arr.shape)
        msk_nustar_cov = nustar_cov_arr*rad_mask
        r500_nustar_cov_frac.append(msk_nustar_cov.sum() / rad_mask.sum())
        
        rel_samp.loc[row_ind, 'NUSTAR_R500_FRAC'] = r500_nustar_cov_frac[row_ind]

### Distribution of coverage of 3.06$^{\circ}$x3.06$^{\circ}$ field centered on MCXC position

In [ ]:
plt.figure(figsize=(7, 6))
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)
plt.hist(field_nustar_cov_frac, bins='auto', color='tab:cyan', alpha=0.7, label='NuSTAR')
plt.ylabel('N', fontsize=15)
plt.xlabel(r'Fraction of coverage of wide field', fontsize=15)

plt.legend(loc='best')
plt.tight_layout()
plt.show()

### Distribution of coverage of MCXC $R_{500}$ centered on MCXC position

In [ ]:
plt.figure(figsize=(7, 6))
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)
plt.hist(r500_nustar_cov_frac, bins='auto', color='tab:cyan', alpha=0.7, label='NuSTAR')

plt.ylabel('N', fontsize=15)
plt.xlabel(r'Fraction of coverage of MCXC $R_{500}$', fontsize=15)

plt.legend(loc='best')
plt.tight_layout()
plt.show()

## No NuSTAR Coverage

The clusters with no NuSTAR observations are all potential targets for this proposal, though we will prioritise based on different criteria. First of all though, there are several clusters which have been found to have planned NuSTAR observations, we will mark them as fully covered in the dataframe (though of course that may not be the case):

### Planned NuSTAR Observations

In [ ]:
planned = [0, 2, 4, 5]

sel = rel_samp['LoVoCCSID'].isin(planned)
rel_samp.loc[sel, 'NUSTAR_R500_FRAC'] = 1

sel = samp['LoVoCCSID'].isin(planned)
samp.loc[sel, 'NUSTAR_R500_FRAC'] = 1

In [ ]:
poss_targ = rel_samp[rel_samp['NUSTAR_R500_FRAC'] < 0.5]
poss_targ.to_csv("poss_nustar_targets.csv", index=False)
poss_targ